
# Citi Bike Data Ingestion (Multi-Month)
Downloads, unzips, normalizes, and saves Citi Bike trip data for a configurable
date range. Handles the schema change that occurred in February 2021.
**Storage estimate (Community Edition — 15GB limit):**
| Range | ~Size | Recommendation |
|---|---|---|
| 1 year | 2–4 GB | ✅ Safe |
| 2 years | 4–8 GB | ✅ Safe |
| 3 years | 6–12 GB | ⚠️ Watch space |
| 4+ years | 10+ GB | ❌ Risk of hitting limit |


## 0. Config — set date range

In [0]:


import pyspark.sql.functions as psf
from pyspark.sql.functions import col, count, avg, max, min, round as spark_round
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, TimestampType
)
import requests, zipfile, os, gc
from io import BytesIO
from datetime import date
from dateutil.relativedelta import relativedelta

In [0]:
START_YEAR  = 2025        # inclusive
START_MONTH = 7
END_YEAR    = 2025        # inclusive
END_MONTH   = 9

# Where to store raw CSVs and the final Delta table on DBFS
RAW_BASE    = "dbfs:/tmp/citibike/raw"        # temp landing zone
DELTA_PATH  = "dbfs:/tmp/citibike/delta/trips"  # permanent Delta table

In [0]:
def month_range(sy, sm, ey, em):
    """Yield (year, month) tuples between two inclusive month boundaries."""
    cur = date(sy, sm, 1)
    end = date(ey, em, 1)
    while cur <= end:
        yield cur.year, cur.month
        cur += relativedelta(months=1)

months = list(month_range(START_YEAR, START_MONTH, END_YEAR, END_MONTH))
print(f"Months to download: {len(months)}")
print(f"First: {months[0]}  |  Last: {months[-1]}")

## 1. Download & extract

In [0]:
# Citi Bike changed their zip filename format a few times.
# We try each known pattern in order and use whichever responds with 200.
URL_PATTERNS = [
    # "https://s3.amazonaws.com/tripdata/{y}{m:02d}-citibike-tripdata.csv.zip",
    "https://s3.amazonaws.com/tripdata/{y}{m:02d}-citibike-tripdata.zip",
    # "https://s3.amazonaws.com/tripdata/{y}{m:02d}-JC-citibike-tripdata.csv.zip",  # Jersey City fallback
]

In [0]:

def build_local_path(year, month):
    local = RAW_BASE.replace("dbfs:", "/dbfs")
    return f"{local}/{year}{month:02d}"

def download_month(year, month):
    out_dir = build_local_path(year, month)
    # Skip if already downloaded (re-run safe)
    if os.path.isdir(out_dir) and os.listdir(out_dir):
        print(f"  {year}-{month:02d} already downloaded, skipping.")
        return True

    os.makedirs(out_dir, exist_ok=True)
    for pattern in URL_PATTERNS:
        url = pattern.format(y=year, m=month)
        try:
            r = requests.get(url, timeout=120)
            if r.status_code == 200:
                with zipfile.ZipFile(BytesIO(r.content)) as zf:
                    zf.extractall(out_dir)
                print(f"  ✅ {year}-{month:02d} downloaded from {url.split('/')[-1]}")
                return True
        except Exception as e:
            print(f"  ⚠️  Pattern failed ({url.split('/')[-1]}): {e}")
            continue

    print(f"  ❌ {year}-{month:02d} — no working URL found, skipping.")
    return False


In [0]:

print("Starting downloads...")
results = {}
for y, m in months:
    results[(y, m)] = download_month(y, m)

ok  = sum(results.values())
bad = len(results) - ok
print(f"\nDone. {ok} months downloaded, {bad} failed.")


## 2. Load & normalize schema

Citi Bike changed column names in **Feb 2021**:

| Pre-2021 (old) | Post-2021 (new) |
|---|---|
| `tripduration` | computed from timestamps |
| `starttime` | `started_at` |
| `stoptime` | `ended_at` |
| `start station id` | `start_station_id` |
| `start station name` | `start_station_name` |
| `start station latitude` | `start_lat` |
| `usertype` (Subscriber/Customer) | `member_casual` (member/casual) |

We normalise everything to the **new schema** below.

In [0]:
NEW_SCHEMA = StructType([
    StructField("ride_id",             StringType()),
    StructField("rideable_type",       StringType()),
    StructField("started_at",          StringType()),   # kept as string; cast later
    StructField("ended_at",            StringType()),
    StructField("start_station_name",  StringType()),
    StructField("start_station_id",    StringType()),
    StructField("end_station_name",    StringType()),
    StructField("end_station_id",      StringType()),
    StructField("start_lat",           DoubleType()),
    StructField("start_lng",           DoubleType()),
    StructField("end_lat",             DoubleType()),
    StructField("end_lng",             DoubleType()),
    StructField("member_casual",       StringType()),
])

In [0]:
# ------------------------------------------------------------------------------------

In [0]:
path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()


In [0]:
path

In [0]:
import zipfile
import os
import requests
from io import BytesIO

curr_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
volume_path = "/Workspace" + path.split('notebooks/')[0] + 'data/'

s3_url = "https://s3.amazonaws.com/tripdata/202508-citibike-tripdata.zip"
# volume_path = "/Workspace/Users/dgc9773@nyu.edu/citibike-analysis/202508"

# Create target directory
os.makedirs(volume_path, exist_ok=True)

# Download the file content into memory
response = requests.get(s3_url)
with zipfile.ZipFile(BytesIO(response.content), 'r') as zip_ref:
    # Extract all contents to the volume path
    zip_ref.extractall(volume_path)


In [0]:
df = spark.read.format("csv").option("header", True).load(volume_path)

In [0]:
df.printSchema()

In [0]:
# get total rides per month (for sept 2025 data)
df.groupBy(psf.year("started_at").alias("year"),psf.month("started_at").alias("month")) \
  .agg(psf.count("*").alias("total_rides")) \
  .orderBy("year", "month") \
  .show()

In [0]:
# get top 10 stations people started at
df.groupBy("start_station_name") \
  .count() \
  .orderBy("count", ascending=False) \
  .limit(10) \
  .show(truncate=False)

In [0]:
# get top 10 stations people ended at
df.groupBy("end_station_name") \
  .count() \
  .orderBy("count", ascending=False) \
  .limit(10) \
  .show(truncate=False)

In [0]:
# get least popular start stations
df.groupBy("start_station_name") \
  .count() \
  .orderBy("count", ascending=True) \
  .limit(10) \
  .show(truncate=False)

In [0]:
# get least popular end stations
df.groupBy("end_station_name") \
  .count() \
  .orderBy("count", ascending=True) \
  .limit(10) \
  .show(truncate=False)

In [0]:

df.withColumn("ride_duration_seconds", (psf.to_timestamp(col("ended_at")) - psf.to_timestamp(col("started_at"))).cast("long")) \
  .select("member_casual", "ride_duration_seconds") \
  .groupBy("member_casual") \
  .agg(psf.round(psf.avg(col("ride_duration_seconds") / 60), 2).alias("avg_duration_min"), 
       psf.round(psf.max(col("ride_duration_seconds") / 60), 2).alias("max_duration_min"), 
       psf.round(psf.min(col("ride_duration_seconds") / 60), 2).alias("min_duration_min")) \
  .show()